<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w09-mcp-first-server/notebook.ipynb)


In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

Colab detected — fetching the course (about 20 seconds)…
ready — the course is at /content/dev3pack


In [2]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w09-e1") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

# Unit 9: Your first MCP server

**Week 0 · Course B, chapter 1 of 3 · about 60 minutes**

**Goal:** Write a real MCP server, connect a client to it over stdio, and make one tool call correctly. Leave knowing that a tool's type hints are its schema and its docstring is its description, because those two are all a model gets before it decides.

**Why it matters:** Session 1 puts *tool* and *MCP server* in the vocabulary table. Session 10 packages a skill. Session 13 connects your client to a hosted MCP surface and asks it to plan a real transaction. This unit is the smallest honest version of all three: forty lines of server, one client, one call.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and read what happens. Every one is a mistake people actually make, and two of them are the two commonest failures in this whole protocol.

**Offline, and honest about it.** The deck calls a public timezone API with `requests`. This notebook computes the conversion locally from `zoneinfo` and a recorded table of twelve zones in January, so nothing here needs the network or a key. The protocol you exercise is the real one; only the arithmetic behind the tool is local.

## 1. The server, and the two things a client cannot see

**Context.** A tool is a function with a decorator. What makes it *usable* is not the code: it is
the signature and the docstring. FastMCP turns the type hints into a JSON Schema, which is what
the client validates against, and it turns the docstring into the tool's description, which is
the only thing a model reads before deciding whether to call it. A tool with no hints has no
schema. A tool with no docstring is a name and nothing else.

This cell writes a working server to a temp directory. It converts correctly. It is also
missing both of the things above.

**Instructions.**

1. Run the cell. The file is written and the tool works. The check still refuses it.
2. Give `from_timezone` a type hint, like the other two parameters.
3. Rewrite the docstring so it names all three arguments with an example each, the way the
   deck's `Args:` block does. The check looks for each parameter name in the docstring.

**Expected output**

```
wrote /tmp/mcp-unit09-.../timezone_server.py
39 lines, one tool: convert_timezone
✅ w09-e1 passed
```

In [3]:
import tempfile
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="mcp-unit09-"))
SERVER_PATH = WORKDIR / "timezone_server.py"
FIXTURE = REPO_ROOT / "units" / "en" / "unit0" / "w09-mcp-first-server" / "fixtures" / "timezones.json"

SERVER_SOURCE = '''
"""A timezone converter, served over MCP."""

import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

from mcp.server.mcpserver import MCPServer

ZONES = json.loads(Path(r"__FIXTURE__").read_text(encoding="utf-8"))["zones"]

mcp = MCPServer("Timezone Converter")


def _zone(name: str):
    """The zone from the system database, or January's offset from the fixture."""
    try:
        from zoneinfo import ZoneInfo

        return ZoneInfo(name)
    except Exception:
        return timezone(timedelta(hours=ZONES[name]))


@mcp.tool()
def convert_timezone(date_time: str, from_timezone: str , to_timezone: str) -> str:  # <------ EDIT THIS LINE
    """Convert a datetime from one timezone to another.
      Args:
        date_time (str): date snd hour format ISO 8601.
            example: "2026-09-19T15:00:00".
        from_timezone (str): source time zone.
            example: "UTC".
        to_timezone (str): destination time zone.
            example: "America/Lima".
      """
    moment = datetime.fromisoformat(date_time).replace(tzinfo=_zone(from_timezone))
    converted = moment.astimezone(_zone(to_timezone)).isoformat()
    return f"Time in {to_timezone}: {converted}"


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

SERVER_PATH.write_text(SERVER_SOURCE.replace("__FIXTURE__", str(FIXTURE)), encoding="utf-8")
print(f"wrote {SERVER_PATH}")
print(f"{len(SERVER_SOURCE.strip().splitlines())} lines, one tool: convert_timezone")

wrote /tmp/mcp-unit09-qsl50_cr/timezone_server.py
41 lines, one tool: convert_timezone


In [4]:
check("w09-e1", SERVER_PATH)

✅ w09-e1 passed


True

## 2. The client, and the commonest failure in MCP

**Context.** A stdio server is not a service you connect to. The client *starts* it, as a child
process, and talks to it over that process's stdin and stdout. So the client needs two things
right: the interpreter to run (`sys.executable`, the one this kernel is using) and the path to
the file. Get the path wrong and the child dies before it can answer, which shows up as a
connection that closes rather than as a missing file.

The starter passes the filename the way you would type it in a terminal that was already sitting
in the right directory. This notebook is not in that directory, and neither is the checker.

**Instructions.**

1. Run the cell. Read the failure: the connection went away, and nothing said "no such file".
2. Point `SERVER_ARGS` at the file itself. `SERVER_PATH` from the last exercise is absolute.
3. Notice `asyncio.wait_for`. A client that waits forever for a server that will never answer is
   a hung notebook, and a hung agent.

**Expected output**

```
 - convert_timezone: Convert a datetime from one timezone to another.
tool_names = ['convert_timezone']
✅ w09-e2 passed
```

**A note on `await`.** The deck writes `asyncio.run(...)`, because a script has no event loop
until it makes one. A notebook already has one running, so `await` is used directly here. Same
code, one less wrapper.

In [17]:
import asyncio
import sys


from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_ARGS = [sys.executable, str(SERVER_PATH)]  # <------ EDIT THIS LINE


async def list_tools(args):
    """Start the server as a child process, initialize, and ask what it provides."""
    params = StdioServerParameters(command=sys.executable, args=args)
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            response = await session.list_tools()
            for tool in response.tools:
                first_line = (tool.description or "").strip().splitlines()[:1]
                print(f" - {tool.name}: {first_line[0] if first_line else '(no description)'}")
            return [tool.name for tool in response.tools]


try:
    tool_names = await asyncio.wait_for(list_tools(SERVER_ARGS), timeout=20)
except Exception as error:
    print(f"the connection failed: {type(error).__name__}: {error}")
    tool_names = []

print(f"tool_names = {tool_names}")

ModuleNotFoundError: No module named 'mcp'

In [16]:
check("w09-e2", (SERVER_PATH, tool_names))

NameError: name 'tool_names' is not defined

## 3. One call, with the argument names the schema actually declares

**Context.** The argument names are not a detail you remember. They are published, in
`tool.inputSchema['properties']`, and reading them there is the difference between a call that
lands and a call that comes back as a validation error. This is the whole of Gecko, the layer
session 13 uses, in one sentence: the surface already says how to call it correctly, and an
agent that guesses instead is the problem being solved.

The starter sends names that read perfectly well to a human and mean nothing to the server.

**Instructions.**

1. Run the cell. The server prints the names it wants, next to the names you sent, and answers
   with a validation error rather than a time.
2. Fix the three keys in `CALL_ARGUMENTS` to match the schema.
3. Read the result. 14:30 in New York in January is the next morning in Tokyo.

**Expected output**

```
the schema wants: ['date_time', 'from_timezone', 'to_timezone']
you are sending: ['date_time', 'from_timezone', 'to_timezone']
Time in Asia/Tokyo: 2025-01-21T04:30:00+09:00
✅ w09-e3 passed
```

In [12]:
CALL_ARGUMENTS = {  # <------ EDIT THESE NAMES
    "date_time": "2025-01-20T14:30:00",
    "from_timezone": "America/New_York",
    "to_timezone": "Asia/Tokyo",
}


async def call_tool(name, arguments):
    """Read the tool's schema, then call the tool and return the text it answered."""
    params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            listed = await session.list_tools()
            schema = listed.tools[0].input_schema
            print(f"the schema wants: {sorted(schema['properties'])}")
            print(f"you are sending: {sorted(arguments)}")
            result = await session.call_tool(name, arguments)
            return result.content[0].text if result.content else ""


try:
    conversion = await asyncio.wait_for(call_tool("convert_timezone", CALL_ARGUMENTS), timeout=20)
except Exception as error:
    conversion = f"the call failed: {type(error).__name__}: {error}"

print(conversion)

the call failed: NameError: name 'StdioServerParameters' is not defined


In [14]:
check("w09-e3", conversion)

❌ w09-e3: the server answered with an error, not a conversion. The argument names come from tool.input_schema['properties'], not from memory; read them off the schema


False

## What to take into unit 10

Three things, and the third is the one that matters beyond this notebook.

| You wrote | It became |
|---|---|
| type hints on a function | the JSON Schema the client validates against |
| a docstring | the description a model reads before it calls |
| an absolute path and a timeout | a client that connects, and one that cannot hang |

Unit 10 adds the other two primitives, resources and prompts, and puts a model in the loop.

Run the scorecard.

In [15]:
review("w09")

w09: 1/3 passed  ·  100/300 marks
   w09-e2: not checked yet; run its check cell
❌ w09-e3: the server answered with an error, not a conversion. The argument names come from tool.input_schema['properties'], not from memory; read them off the schema


False